# Laboratório — Desenho experimental e testes A/B

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/joaopaulomirandamatias/ai-lab/blob/main/02-statistics/notebooks/20-desenho-experimental-ab-laboratorio.ipynb)

Este laboratório reproduz um teste A/B de uma política de IA, do planejamento aos diagnósticos. Os dados são sintéticos e nenhuma célula depende de download.

**Dependências:** Python 3.11+, NumPy 2.0+, pandas 2.0+, SciPy 1.12+ e Matplotlib 3.8+.

**Reprodutibilidade:** seed global `20260907`; versões são impressas; verificações finais falham explicitamente se um resultado metodológico deixar de ocorrer.

## Roteiro

1. congelar uma especificação mínima;
2. planejar amostra por MDE e poder;
3. randomizar dentro de blocos;
4. estimar o efeito por intenção de tratar;
5. validar SRM e testes A/A;
6. observar a inflação causada por parada opcional;
7. respeitar dependência por usuário;
8. comparar dois sistemas de IA de modo pareado por tarefa.

In [ ]:
import sys
import numpy as np
import pandas as pd
import scipy
from scipy import special, stats
import matplotlib
import matplotlib.pyplot as plt

SEED = 20260907
rng = np.random.default_rng(SEED)

print(f"Python {sys.version.split()[0]}")
print(f"NumPy {np.__version__} | pandas {pd.__version__} | SciPy {scipy.__version__} | Matplotlib {matplotlib.__version__}")
print(f"Seed: {SEED}")

## 1. Congele o protocolo antes de olhar o efeito

Uma estrutura versionada não substitui um protocolo, mas torna explícitas as decisões confirmatórias. O MDE abaixo é uma diferença **absoluta** de 1 ponto percentual.

In [ ]:
spec = {
    "pergunta": "Oferecer a política B aumenta o sucesso em sete dias?",
    "populacao": "usuários elegíveis de web, mobile e API",
    "unidade_randomizacao": "usuario",
    "tratamento": "política B de recuperação; A é a política atual",
    "metrica_primaria": "ao menos uma tarefa correta em sete dias",
    "guardrails": ["incidente de segurança", "custo por tarefa", "latência p95"],
    "estimando": "diferença ITT B - A na população elegível",
    "taxa_base": 0.10,
    "mde_absoluto": 0.01,
    "alpha": 0.05,
    "poder": 0.80,
    "regra_parada": "tamanho planejado e ao menos dois ciclos semanais",
}

for chave, valor in spec.items():
    print(f"{chave}: {valor}")

## 2. Planejamento para duas proporções

Usaremos o tamanho de efeito angular de Cohen,

\[
h=2\arcsin\sqrt{p_B}-2\arcsin\sqrt{p_A},
\]

e a aproximação normal para grupos iguais:

\[
n\approx 2\left(\frac{z_{1-\alpha/2}+z_{1-\beta}}{h}\right)^2.
\]

Planejamento é uma aproximação: perdas, agrupamento, atraso do desfecho e sazonalidade ainda precisam ser incorporados.

In [ ]:
def cohen_h(p_a, p_b):
    return 2 * np.arcsin(np.sqrt(p_b)) - 2 * np.arcsin(np.sqrt(p_a))


def n_por_grupo_duas_proporcoes(p_a, p_b, alpha=0.05, power=0.80):
    h = abs(cohen_h(p_a, p_b))
    z_alpha = stats.norm.ppf(1 - alpha / 2)
    z_power = stats.norm.ppf(power)
    return int(np.ceil(2 * ((z_alpha + z_power) / h) ** 2))


p_a_planejado = spec["taxa_base"]
p_b_planejado = p_a_planejado + spec["mde_absoluto"]
n_planejado = n_por_grupo_duas_proporcoes(
    p_a_planejado, p_b_planejado, spec["alpha"], spec["poder"]
)

for delta in [0.005, 0.010, 0.020]:
    n = n_por_grupo_duas_proporcoes(0.10, 0.10 + delta)
    print(f"MDE={100*delta:.1f} pp -> {n:,} unidades por grupo")
print(f"\nPlano principal: {n_planejado:,} por grupo; total={2*n_planejado:,}")

In [ ]:
deltas = np.linspace(0.004, 0.025, 100)
ns = np.array([n_por_grupo_duas_proporcoes(0.10, 0.10 + d) for d in deltas])

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(100 * deltas, ns, color="#6f42c1", linewidth=2)
ax.axvline(1.0, color="#d62728", linestyle="--", label="MDE do protocolo: 1 pp")
ax.set(xlabel="MDE absoluto (pontos percentuais)", ylabel="Unidades por grupo",
       title="Efeitos menores exigem muito mais amostra")
ax.grid(alpha=0.25)
ax.legend()
plt.tight_layout()
plt.show()

## 3. Randomização bloqueada

Canal e faixa de atividade existem antes do tratamento e são prognósticos. Randomizamos dentro de cada combinação para evitar grandes desequilíbrios. A análise principal continua sendo ITT: cada pessoa permanece no grupo sorteado.

In [ ]:
n_total = 2 * n_planejado
df = pd.DataFrame({
    "usuario_id": np.arange(n_total),
    "canal": rng.choice(["web", "mobile", "api"], n_total, p=[0.50, 0.35, 0.15]),
    "atividade": rng.choice(["baixa", "alta"], n_total, p=[0.65, 0.35]),
})

df["grupo"] = ""
for _, idx in df.groupby(["canal", "atividade"]).groups.items():
    idx = np.array(list(idx))
    rng.shuffle(idx)
    df.loc[idx[: len(idx) // 2], "grupo"] = "A"
    df.loc[idx[len(idx) // 2 :], "grupo"] = "B"

balance = pd.crosstab([df["canal"], df["atividade"]], df["grupo"])
balance["diferença"] = balance.get("B", 0) - balance.get("A", 0)
print(balance)
print("\nTamanho por grupo:", df["grupo"].value_counts().to_dict())

## 4. Desfecho sintético e análise ITT

A probabilidade de base varia por canal e atividade. O tratamento acrescenta 1,2 ponto percentual. Essa heterogeneidade não invalida a randomização; o bloqueio melhora o equilíbrio.

In [ ]:
base_canal = df["canal"].map({"mobile": 0.08, "web": 0.11, "api": 0.16}).astype(float)
bonus_atividade = df["atividade"].map({"baixa": 0.00, "alta": 0.035}).astype(float)
efeito_real = 0.012
prob_sucesso = base_canal + bonus_atividade + efeito_real * (df["grupo"] == "B")
df["sucesso"] = rng.binomial(1, prob_sucesso)

resumo = df.groupby("grupo")["sucesso"].agg(["sum", "count", "mean"])
p_a = resumo.loc["A", "mean"]
p_b = resumo.loc["B", "mean"]
n_a = int(resumo.loc["A", "count"])
n_b = int(resumo.loc["B", "count"])
delta = p_b - p_a
lift = delta / p_a
se = np.sqrt(p_a * (1-p_a) / n_a + p_b * (1-p_b) / n_b)
ci = (delta - stats.norm.ppf(0.975) * se, delta + stats.norm.ppf(0.975) * se)
p_pool = (resumo["sum"].sum()) / (n_a + n_b)
se_h0 = np.sqrt(p_pool * (1-p_pool) * (1/n_a + 1/n_b))
z = delta / se_h0
p_value = 2 * stats.norm.sf(abs(z))

print(resumo.to_string(float_format=lambda x: f"{x:.6f}"))
print(f"\nEfeito absoluto B-A = {100*delta:.4f} pp")
print(f"Lift relativo = {100*lift:.2f}%")
print(f"IC 95% = [{100*ci[0]:.4f}; {100*ci[1]:.4f}] pp")
print(f"z = {z:.6f}; p-value = {p_value:.6g}")

## 5. SRM antes do efeito

O teste qui-quadrado compara contagens observadas às proporções de atribuição planejadas. Um p-value extremamente pequeno é um alarme de integridade, não evidência de que B funciona.

In [ ]:
def srm_pvalue(contagens, proporcoes):
    contagens = np.asarray(contagens)
    esperadas = contagens.sum() * np.asarray(proporcoes) / np.sum(proporcoes)
    return stats.chisquare(contagens, f_exp=esperadas)


contagens_validas = df["grupo"].value_counts().reindex(["A", "B"]).to_numpy()
srm_valido = srm_pvalue(contagens_validas, [0.5, 0.5])
srm_falho = srm_pvalue([10_800, 9_200], [0.5, 0.5])

print(f"Experimento simulado: contagens={contagens_validas.tolist()}, p_SRM={srm_valido.pvalue:.6f}")
print(f"Falha ilustrativa: contagens=[10800, 9200], p_SRM={srm_falho.pvalue:.3e}")

## 6. Testes A/A e calibração

Em cada repetição, A e A recebem a mesma taxa real de 10%. Se a implementação está calibrada, aproximadamente 5% dos testes fixos rejeitam ao nível de 5%.

In [ ]:
n_aa = 1_000
n_sim = 10_000
x_a = rng.binomial(n_aa, 0.10, size=n_sim)
x_b = rng.binomial(n_aa, 0.10, size=n_sim)
p1, p2 = x_a / n_aa, x_b / n_aa
p_pool_aa = (x_a + x_b) / (2 * n_aa)
se_aa = np.sqrt(p_pool_aa * (1-p_pool_aa) * (2/n_aa))
z_aa = np.divide(p2-p1, se_aa, out=np.zeros_like(p1), where=se_aa > 0)
p_aa = 2 * stats.norm.sf(np.abs(z_aa))
taxa_falso_positivo_aa = np.mean(p_aa < 0.05)

print(f"Repetições A/A: {n_sim:,}")
print(f"Falsos positivos em alpha=5%: {taxa_falso_positivo_aa:.4f}")
print(f"Mediana dos p-values: {np.median(p_aa):.4f}")

## 7. Parada opcional

Agora ambos os grupos continuam idênticos, mas consultamos o p-value após cada lote e marcamos como “descoberta” qualquer experimento que cruze 0,05 em uma das dez consultas. Isso **não** é um desenho sequencial válido.

In [ ]:
n_experimentos = 5_000
n_looks = 10
lote_por_grupo = 400
inc_a = rng.binomial(lote_por_grupo, 0.10, size=(n_experimentos, n_looks))
inc_b = rng.binomial(lote_por_grupo, 0.10, size=(n_experimentos, n_looks))
cum_a, cum_b = np.cumsum(inc_a, axis=1), np.cumsum(inc_b, axis=1)
n_acum = lote_por_grupo * np.arange(1, n_looks+1)
pa, pb = cum_a / n_acum, cum_b / n_acum
pool = (cum_a + cum_b) / (2 * n_acum)
se0 = np.sqrt(pool * (1-pool) * (2/n_acum))
z_looks = np.divide(pb-pa, se0, out=np.zeros_like(pa), where=se0 > 0)
p_looks = 2 * stats.norm.sf(np.abs(z_looks))
fp_acumulado = np.mean(np.minimum.accumulate(p_looks, axis=1) < 0.05, axis=0)
fp_final_fixo = np.mean(p_looks[:, -1] < 0.05)
fp_parada_opcional = np.mean(np.any(p_looks < 0.05, axis=1))

print(f"Teste apenas no fim: {fp_final_fixo:.4f}")
print(f"Parar na primeira significância: {fp_parada_opcional:.4f}")

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(np.arange(1, n_looks+1), fp_acumulado, marker="o", color="#d62728")
ax.axhline(0.05, linestyle="--", color="#333333", label="alvo nominal de 5%")
ax.set(xlabel="Número de consultas", ylabel="Probabilidade acumulada de falso positivo",
       title="Consultar repetidamente um teste fixo infla o erro tipo I")
ax.set_xticks(np.arange(1, n_looks+1))
ax.grid(alpha=0.25)
ax.legend()
plt.tight_layout()
plt.show()

## 8. Sessões não são usuários independentes

O tratamento é sorteado por usuário. Cada pessoa gera 20 sessões correlacionadas por um componente latente. O erro-padrão ingênuo por sessão será comparado ao erro-padrão entre médias de usuários.

In [ ]:
n_users = 800
sessoes_por_user = 20
grupo_user = np.repeat(np.array([0, 1]), n_users // 2)
rng.shuffle(grupo_user)
efeito_usuario = rng.normal(0, 1.0, n_users)

latencia = (
    5.0
    + np.repeat(efeito_usuario, sessoes_por_user)
    + rng.normal(0, 0.5, n_users * sessoes_por_user)
)
long = pd.DataFrame({
    "usuario": np.repeat(np.arange(n_users), sessoes_por_user),
    "grupo": np.repeat(grupo_user, sessoes_por_user),
    "latencia": latencia,
})

def diff_e_se_indep(y, g):
    a, b = y[g == 0], y[g == 1]
    diff = b.mean() - a.mean()
    se = np.sqrt(a.var(ddof=1)/len(a) + b.var(ddof=1)/len(b))
    return diff, se


efeito_naive, se_naive = diff_e_se_indep(long["latencia"].to_numpy(), long["grupo"].to_numpy())
por_usuario = long.groupby(["usuario", "grupo"], as_index=False)["latencia"].mean()
efeito_cluster, se_cluster = diff_e_se_indep(
    por_usuario["latencia"].to_numpy(), por_usuario["grupo"].to_numpy()
)

print(f"Efeito (mesma estimativa) = {efeito_cluster:.6f}")
print(f"SE ingênuo por sessão = {se_naive:.6f}")
print(f"SE respeitando usuário = {se_cluster:.6f}")
print(f"Subestimação: SE correto é {se_cluster/se_naive:.2f} vezes maior")

## 9. Sistemas de IA comparados nas mesmas tarefas

As tarefas têm dificuldades diferentes. A e B respondem aos mesmos itens; portanto, calculamos diferenças pareadas e contamos discordâncias. O teste binomial sobre discordâncias é a forma exata do teste de McNemar sem correção.

In [ ]:
n_tasks = 2_000
dificuldade = rng.normal(0, 1, n_tasks)
p_modelo_a = special.expit(-0.35 - 0.85 * dificuldade)
p_modelo_b = np.clip(p_modelo_a + 0.04, 0, 1)

# Cópula gaussiana: resultados correlacionados dentro da mesma tarefa.
comum = rng.normal(size=n_tasks)
u_a = stats.norm.cdf(np.sqrt(0.60)*comum + np.sqrt(0.40)*rng.normal(size=n_tasks))
u_b = stats.norm.cdf(np.sqrt(0.60)*comum + np.sqrt(0.40)*rng.normal(size=n_tasks))
y_a = (u_a < p_modelo_a).astype(int)
y_b = (u_b < p_modelo_b).astype(int)
d = y_b - y_a

ganha_b = int(np.sum((y_b == 1) & (y_a == 0)))
ganha_a = int(np.sum((y_b == 0) & (y_a == 1)))
efeito_pareado = d.mean()
se_pareado = d.std(ddof=1) / np.sqrt(n_tasks)
se_indep = np.sqrt(y_a.var(ddof=1)/n_tasks + y_b.var(ddof=1)/n_tasks)
p_mcnemar = stats.binomtest(ganha_b, ganha_a + ganha_b, p=0.5).pvalue

print(f"Acurácia A={y_a.mean():.4f}; B={y_b.mean():.4f}; B-A={efeito_pareado:.4f}")
print(f"Discordâncias: B vence={ganha_b}; A vence={ganha_a}")
print(f"SE pareado={se_pareado:.6f}; SE independente={se_indep:.6f}")
print(f"McNemar exato p={p_mcnemar:.6g}")

## 10. Verificações metodológicas

Os limites abaixo não exigem valores idênticos em versões futuras, mas garantem que os fenômenos didáticos continuem presentes.

In [ ]:
assert 14_000 < n_planejado < 16_000
assert balance["diferença"].abs().max() <= 1
assert abs(delta - efeito_real) < 0.012
assert srm_valido.pvalue > 0.10
assert srm_falho.pvalue < 1e-20
assert 0.035 < taxa_falso_positivo_aa < 0.065
assert 0.035 < fp_final_fixo < 0.065
assert fp_parada_opcional > 0.12
assert se_cluster / se_naive > 2.5
assert efeito_pareado > 0.01
assert se_pareado < se_indep

print("Todas as verificações foram aprovadas.")
print(f"Amostra planejada: {n_planejado:,} por grupo")
print(f"Efeito ITT observado: {100*delta:.4f} pp; IC=[{100*ci[0]:.4f}; {100*ci[1]:.4f}] pp")
print(f"A/A: falso positivo={taxa_falso_positivo_aa:.4f}")
print(f"Parada opcional: falso positivo={fp_parada_opcional:.4f}")
print(f"Dependência: inflação necessária do SE={se_cluster/se_naive:.2f}x")
print(f"IA pareada: diferença={100*efeito_pareado:.2f} pp; p={p_mcnemar:.6g}")

## Conclusões

- A amostra nasce do efeito relevante e do poder, não do p-value observado.
- Bloquear antes da atribuição preserva equilíbrio sem escolher grupos pelo resultado.
- SRM e exposição são portões de integridade anteriores à análise de eficácia.
- Testes A/A ajudam a calibrar a plataforma.
- Consultar repetidamente um teste de horizonte fixo infla falsos positivos.
- A incerteza deve respeitar o nível de randomização e a dependência entre observações.
- Em IA, comparar sistemas nas mesmas tarefas cria um experimento pareado mais informativo.

Volte à [Aula 20](../aulas/20-desenho-experimental-ab.md) para o checklist completo e as referências.